# BankLoan AI — Model Training Notebook
**Dataset:** Give Me Some Credit (lihxlhx/give-me-some-credit)
**Kaggle Link:** https://www.kaggle.com/datasets/lihxlhx/give-me-some-credit

This notebook trains LightGBM + XGBoost + Random Forest ensemble, saves all models as .pkl files.
Download the output models and place them in `backend/models/`.

In [ ]:
# Install required packages
!pip install lightgbm xgboost shap imbalanced-learn optuna --quiet

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve
)
from sklearn.pipeline import Pipeline

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42)

os.makedirs('/kaggle/working/models', exist_ok=True)
print('Setup complete.')

## 1. Load Dataset

In [ ]:
import glob

# Auto-detect the correct CSV path
print("Files in /kaggle/input/:")
for f in glob.glob('/kaggle/input/**/*', recursive=True):
    print(f)

# Find cs-training.csv wherever it is
matches = glob.glob('/kaggle/input/**/cs-training.csv', recursive=True)
if not matches:
    # Some uploads name it differently
    matches = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    print("\nNo cs-training.csv found. All CSVs:", matches)
else:
    DATA_PATH = matches[0]
    print(f"\nUsing: {DATA_PATH}")

DATA_PATH = matches[0]
df = pd.read_csv(DATA_PATH, index_col=0)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Target distribution:\n{df["SeriousDlqin2yrs"].value_counts(normalize=True).round(3)}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

## 2. Preprocessing & Feature Engineering

In [ ]:
class LoanPreprocessor:
    def __init__(self):
        self.income_median = None
        self.age_median = None
        self.iqr_bounds = {}
        self.feature_columns = None

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Fill missing values
        self.income_median = df['MonthlyIncome'].median()
        self.age_median = df['age'].median()
        df['MonthlyIncome'] = df['MonthlyIncome'].fillna(self.income_median)
        df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)
        df['age'] = df['age'].fillna(self.age_median).clip(18, 100).astype(int)

        # Cap outliers using IQR (1.5x)
        numeric_cols = [
            'RevolvingUtilizationOfUnsecuredLines', 'DebtRatio',
            'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
            'NumberRealEstateLoansOrLines', 'NumberOfDependents'
        ]
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.iqr_bounds[col] = (lower, upper)
            df[col] = df[col].clip(lower, upper)

        df = self._engineer_features(df)
        self.feature_columns = [c for c in df.columns if c != 'SeriousDlqin2yrs']
        return df

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['MonthlyIncome'] = df['MonthlyIncome'].fillna(self.income_median)
        df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)
        df['age'] = df['age'].fillna(self.age_median).clip(18, 100).astype(int)

        for col, (lower, upper) in self.iqr_bounds.items():
            if col in df.columns:
                df[col] = df[col].clip(lower, upper)

        df = self._engineer_features(df)
        return df

    def _engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df['DebtToIncome'] = df['DebtRatio'] * df['MonthlyIncome']

        late_cols = [
            'NumberOfTime30-59DaysPastDueNotWorse',
            'NumberOfTime60-89DaysPastDueNotWorse',
            'NumberOfTimes90DaysLate'
        ]
        df['LatePaymentTotal'] = df[late_cols].sum(axis=1)
        df['CreditUtilizationRisk'] = df['RevolvingUtilizationOfUnsecuredLines']
        df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

        age_bins = [0, 25, 35, 50, 65, 200]
        age_labels = [0, 1, 2, 3, 4]
        df['AgeGroup'] = pd.cut(
            df['age'], bins=age_bins, labels=age_labels
        ).cat.add_categories([-1]).fillna(-1).astype(int)

        return df


preprocessor = LoanPreprocessor()
df_processed = preprocessor.fit_transform(df)
print(f'Processed shape: {df_processed.shape}')
print('New features:', ['DebtToIncome', 'LatePaymentTotal', 'CreditUtilizationRisk', 'IncomePerDependent', 'AgeGroup'])

## 3. Train/Test Split & SMOTE

In [ ]:
TARGET = 'SeriousDlqin2yrs'
FEATURE_COLS = [c for c in df_processed.columns if c != TARGET]

X = df_processed[FEATURE_COLS]
y = df_processed[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Class distribution before SMOTE: {y_train.value_counts().to_dict()}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Class distribution after SMOTE: {pd.Series(y_train_bal).value_counts().to_dict()}')
print(f'Balanced train shape: {X_train_bal.shape}')

## 4. Train Models

In [ ]:
# Class weight ratio for XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
ratio_neg_to_pos = n_neg / n_pos
print(f'scale_pos_weight for XGBoost: {ratio_neg_to_pos:.2f}')

In [ ]:
print('Training LightGBM...')
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(X_train_bal, y_train_bal)
lgbm_auc = roc_auc_score(y_test, lgbm.predict_proba(X_test)[:, 1])
print(f'LightGBM Test AUC: {lgbm_auc:.4f}')

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_bal, y_train_bal)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest Test AUC: {rf_auc:.4f}')

In [ ]:
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    scale_pos_weight=ratio_neg_to_pos,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False
)
xgb.fit(X_train_bal, y_train_bal)
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f'XGBoost Test AUC: {xgb_auc:.4f}')

In [ ]:
print('Training Ensemble (Soft Voting)...')
ensemble = VotingClassifier(
    estimators=[('lgbm', lgbm), ('rf', rf), ('xgb', xgb)],
    voting='soft',
    weights=[2, 1, 1]
)
ensemble.fit(X_train_bal, y_train_bal)
ensemble_auc = roc_auc_score(y_test, ensemble.predict_proba(X_test)[:, 1])
print(f'Ensemble Test AUC: {ensemble_auc:.4f}')

## 5. Metrics

In [ ]:
def compute_ks(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return round(float(np.max(tpr - fpr)), 4)

y_prob_ensemble = ensemble.predict_proba(X_test)[:, 1]
y_pred_ensemble = (y_prob_ensemble >= 0.35).astype(int)

auc_score = roc_auc_score(y_test, y_prob_ensemble)
gini = 2 * auc_score - 1
ks = compute_ks(y_test, y_prob_ensemble)

cm = confusion_matrix(y_test, y_pred_ensemble)
tn, fp, fn, tp = cm.ravel()

metrics = {
    'test_auc':  round(auc_score, 4),
    'gini':      round(gini, 4),
    'ks':        ks,
    'precision': round(precision_score(y_test, y_pred_ensemble), 4),
    'recall':    round(recall_score(y_test, y_pred_ensemble), 4),
    'f1':        round(f1_score(y_test, y_pred_ensemble), 4),
    'confusion_matrix': {'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)}
}

print('=== Ensemble Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Model comparison
comparison = pd.DataFrame({
    'Model': ['LightGBM', 'Random Forest', 'XGBoost', 'Ensemble'],
    'AUC':   [lgbm_auc, rf_auc, xgb_auc, ensemble_auc]
})
comparison['Gini'] = 2 * comparison['AUC'] - 1
print(comparison.to_string(index=False))

## 6. Feature Importance

In [ ]:
feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='Blues_r')
plt.title('LightGBM Feature Importance (Top 15)')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance.png', dpi=150)
plt.show()
print(feat_imp)

## 7. SHAP Analysis

In [ ]:
print('Computing SHAP values (sample of 500 rows)...')
explainer = shap.TreeExplainer(lgbm)
X_sample = X_test.sample(500, random_state=42)
shap_values = explainer.shap_values(X_sample)

# shap_values[1] = class 1 (default) SHAP values
shap_class1 = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_class1, X_sample, feature_names=FEATURE_COLS, show=False)
plt.title('SHAP Summary — Default Risk Drivers')
plt.tight_layout()
plt.savefig('/kaggle/working/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save All Models & Artifacts

In [ ]:
MODELS_DIR = '/kaggle/working/models'

joblib.dump(lgbm,       f'{MODELS_DIR}/lgbm_model.pkl')
joblib.dump(rf,         f'{MODELS_DIR}/rf_model.pkl')
joblib.dump(xgb,        f'{MODELS_DIR}/xgb_model.pkl')
joblib.dump(ensemble,   f'{MODELS_DIR}/ensemble_model.pkl')
joblib.dump(preprocessor, f'{MODELS_DIR}/preprocessor.pkl')

# Save X_train for OOD detection
joblib.dump(X_train_bal, f'{MODELS_DIR}/X_train_reference.pkl')

# Save feature column list
joblib.dump(FEATURE_COLS, f'{MODELS_DIR}/feature_columns.pkl')

# Save metrics as JSON
import json
metrics['feature_importances'] = feat_imp.set_index('feature')['importance'].to_dict()
with open(f'{MODELS_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved files:')
for f in sorted(os.listdir(MODELS_DIR)):
    size_kb = os.path.getsize(f'{MODELS_DIR}/{f}') / 1024
    print(f'  {f:35s}  {size_kb:.1f} KB')

In [ ]:
# Zip all models for easy download
import shutil
shutil.make_archive('/kaggle/working/bankloan_models', 'zip', MODELS_DIR)
zip_size = os.path.getsize('/kaggle/working/bankloan_models.zip') / (1024 * 1024)
print(f'bankloan_models.zip created — {zip_size:.1f} MB')
print('Download this zip from Kaggle Output tab and extract to backend/models/')

## 9. Quick Sanity Check — Single Prediction

In [ ]:
# Test a sample prediction end-to-end
sample_customer = {
    'RevolvingUtilizationOfUnsecuredLines': 0.3,
    'age': 42,
    'NumberOfTime30-59DaysPastDueNotWorse': 0,
    'DebtRatio': 0.35,
    'MonthlyIncome': 6000.0,
    'NumberOfOpenCreditLinesAndLoans': 5,
    'NumberOfTimes90DaysLate': 0,
    'NumberRealEstateLoansOrLines': 1,
    'NumberOfTime60-89DaysPastDueNotWorse': 0,
    'NumberOfDependents': 2
}

sample_df = pd.DataFrame([sample_customer])
sample_processed = preprocessor.transform(sample_df)
sample_processed = sample_processed[FEATURE_COLS]

prob = ensemble.predict_proba(sample_processed)[0][1]
approval_prob = 1 - prob  # prob of default → invert for approval
risk_score = int(300 + approval_prob * 600)

if approval_prob >= 0.65:
    decision = 'APPROVE'
elif approval_prob < 0.35:
    decision = 'REJECT'
else:
    decision = 'MANUAL_REVIEW'

print(f'Default probability:  {prob:.3f}')
print(f'Approval probability: {approval_prob:.3f}')
print(f'Risk score:           {risk_score} / 900')
print(f'Decision:             {decision}')

## Done!

### Next steps:
1. Download `bankloan_models.zip` from the **Output** tab
2. Extract it → you get 7 files
3. Place all `.pkl` files into `backend/models/`
4. Place `metrics.json` into `backend/models/`
5. Run `uvicorn main:app --reload --port 8000`

### Expected metrics:
| Metric | Expected |
|--------|----------|
| AUC | 0.855 – 0.870 |
| Gini | 0.710 – 0.740 |
| KS Statistic | 0.55 – 0.65 |